# Folder 03 / file 01 — approval_check (read Approved only)

UCI Adult Census Income ([dataset](https://archive.ics.uci.edu/dataset/2/adult)): binary target `income_gt_50k` where **`>50K` = 1** and **`<=50K` = 0**. Fourteen census features; official split is `adult.data` (train) / `adult.test` (holdout).

Inlined replica of `src/n03_prod_cutover/n01_approval.py`. Run cells **in order** (local or Jobs). Catalog/schema/model/mode come from task env.

Read-only approval gate for the Adult dest @challenger version.


## 1 — Imports


In [ ]:
from src.n00_shared.protocol import challenger_pin_matches, check_approval_read
from src.n00_shared.runtime import Settings, configure_mlflow, load_settings, mlflow_client, operator_param, set_task_value


## 2 — `_tag_map`


In [ ]:
def _tag_map(client, name: str, version: str) -> dict:
    mv = client.get_model_version(name, version)
    tags = mv.tags or {}
    if isinstance(tags, dict):
        return tags
    return {t.key: t.value for t in tags}


## 3 — `resolve_cutover_dest_version`


In [ ]:
def resolve_cutover_dest_version(settings: Settings) -> str:
    pin = operator_param("source_model_version", "").strip()
    if not pin:
        raise RuntimeError("cutover requires the same source_model_version pin as the gate")
    client = mlflow_client()
    challenger = client.get_model_version_by_alias(settings.dest_model_name, "challenger")
    tags = _tag_map(client, settings.dest_model_name, challenger.version)
    if not challenger_pin_matches(tags.get("source_model_version"), pin):
        raise RuntimeError("dest @challenger source_model_version tag does not match the pin")
    return str(challenger.version)


## 4 — `settings = load_settings()`


In [ ]:
settings = load_settings()


## 5 — `run()` step 1/1


In [ ]:
configure_mlflow(settings)
version = resolve_cutover_dest_version(settings)
client = mlflow_client()
tags = _tag_map(client, settings.dest_model_name, version)
check_approval_read(
    tags.get("approval_check"),
    tags.get("approved_by"),
    settings.approver_identities,
    settings.job_run_as_sp,
)
set_task_value("model_version", version)
print(f"approval_check read-only ok dest v{version}")
